# Code mẫu — Training loop

Một vòng lặp train làm bốn việc theo đúng thứ tự: lấy batch, tính gradient, cập nhật tham số, rồi đánh giá. Ví dụ đầu soi kỹ một bước duy nhất. Ví dụ thứ hai chạy trọn một quy trình có validation, chọn checkpoint và khôi phục model. Ví dụ cuối kiểm cách báo loss khi batch cuối nhỏ hơn các batch khác.

## Ví dụ 1: Nhìn tham số ở ba thời điểm của một batch

Có năm mẫu hồi quy theo quy tắc $y=2x$. Với batch size 2, ta có ba batch, và batch cuối chỉ có một mẫu. Ta lấy batch đầu, rồi xem trọng số trước `backward`, sau `backward`, và sau `step`.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)
x = torch.arange(1., 6.).unsqueeze(1)
y = 2 * x
loader = DataLoader(TensorDataset(x, y), batch_size=2, shuffle=False)
print("kích thước các batch:", [len(xb) for xb, _ in loader])
weight = torch.nn.Parameter(torch.tensor(1.))
optimizer = torch.optim.SGD([weight], lr=.1)
xb, yb = next(iter(loader))
optimizer.zero_grad(set_to_none=True)
prediction = xb * weight
assert prediction.shape == yb.shape
loss = (prediction - yb).square().mean()
before = weight.item()
loss.backward()
print("loss, gradient:", loss.item(), weight.grad.item())
print("trước / sau backward:", before, weight.item())
optimizer.step()
print("sau step:", weight.item())
assert before == 1. and weight.grad.item() == -5. and weight.item() == 1.5

kích thước các batch: [2, 2, 1]
loss, gradient: 2.5 -5.0
trước / sau backward: 1.0 1.0
sau step: 1.5


Batch đầu có $x=1$ và $x=2$. Với trọng số 1, dự đoán là 1 và 2, còn target là 2 và 4. Loss bằng $(1+4)/2=2.5$, và gradient bằng $2[(1-2)\cdot1+(2-4)\cdot2]/2=-5$.

`backward` chỉ tính ra gradient, trọng số vẫn đứng yên ở 1. Phải tới `step`, trọng số mới đổi thành $1-0.1\cdot(-5)=1.5$. Duyệt hết loader, một epoch sẽ có ba lần `step` như thế.

## Ví dụ 2: Chia tập, chọn checkpoint, rồi suy luận từ trạng thái đã lưu

Dữ liệu hai chiều có nhãn theo quy tắc $x_1+0.5x_2>0$. Mười hai mẫu train và bốn mẫu validation được tách sẵn, không có điểm nào trùng. Bước chuẩn hóa chỉ dùng mean và std của train. Bài toán nhỏ và biết trước quy tắc, nên ta tập trung kiểm quy trình, không phải kiểm model.

In [ ]:
import copy
import io
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)
torch.manual_seed(41)
raw_train = torch.tensor([[-2., -1.], [-1., -1.], [-1., 1.], [-.5, -.5],
                          [.5, -1.], [.5, .5], [1., -1.], [1., 1.],
                          [2., 1.], [2., -1.], [-2., 1.], [-.5, 1.]])
raw_valid = torch.tensor([[-1.5, 0.], [-.5, 0.], [.5, 0.], [1.5, 0.]])
train_y = (raw_train[:, 0] + .5 * raw_train[:, 1] > 0).long()
valid_y = (raw_valid[:, 0] + .5 * raw_valid[:, 1] > 0).long()
mean = raw_train.mean(dim=0)
scale = raw_train.std(dim=0, unbiased=False).clamp_min(1e-6)
train_x, valid_x = (raw_train - mean) / scale, (raw_valid - mean) / scale
train_data = TensorDataset(train_x, train_y)
loader = DataLoader(train_data, batch_size=5, shuffle=True, num_workers=0,
                    generator=torch.Generator().manual_seed(41))
model = nn.Linear(2, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=.2)
print("train / validation:", len(train_y), len(valid_y))
print("mean train:", [round(v, 4) for v in mean.tolist()])
print("batch mỗi epoch:", len(loader))

train / validation: 12 4
mean train: [0.0, 0.0]
batch mỗi epoch: 3


Mỗi epoch có ba batch, gồm 5, 5 và 2 mẫu. Validation không được tính mean và std riêng. Dữ liệu mới nào cũng phải đi qua đúng phép biến đổi đã học từ train.

Tiếp theo, ta viết riêng một hàm đánh giá. Cuối mỗi epoch, hàm này chấm cả train lẫn validation bằng cùng một model, ở chế độ `eval`. Ta in thêm loss ghi được trong lúc train để so. Loss trong lúc train được cộng dồn qua từng batch, trong khi trọng số vẫn đang thay đổi.

In [ ]:
def evaluate(x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = nn.functional.cross_entropy(logits, y).item()
        accuracy = (logits.argmax(dim=1) == y).float().mean().item()
    return loss, accuracy

best_loss, best_epoch, best_state = float("inf"), 0, None
print("epoch | loss trong loop | train eval | valid loss | valid accuracy")
for epoch in range(1, 31):
    model.train()
    total, seen = 0., 0
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        assert logits.shape == (len(yb), 2) and yb.dtype == torch.long
        loss = nn.functional.cross_entropy(logits, yb)
        assert torch.isfinite(loss)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
        seen += len(yb)
    train_loss, _ = evaluate(train_x, train_y)
    valid_loss, valid_accuracy = evaluate(valid_x, valid_y)
    if valid_loss < best_loss:
        best_loss, best_epoch = valid_loss, epoch
        best_state = copy.deepcopy(model.state_dict())
    if epoch in [1, 5, 15, 30]:
        print(epoch, *[round(v, 4) for v in
                       [total / seen, train_loss, valid_loss, valid_accuracy]])
model.load_state_dict(best_state)
print("epoch được chọn:", best_epoch)
print("validation sau khôi phục:", tuple(round(v, 4) for v in evaluate(valid_x, valid_y)))
assert abs(evaluate(valid_x, valid_y)[0] - best_loss) < 1e-7

epoch | loss trong loop | train eval | valid loss | valid accuracy
1 0.5442 0.4438 0.4064 0.75
5 0.3102 0.2843 0.2706 1.0
15 0.1923 0.1873 0.1974 1.0
30 0.1495 0.1438 0.1699 1.0
epoch được chọn: 30
validation sau khôi phục: (0.1699, 1.0)


Ở epoch 1, loss trong lúc train là 0.544, còn chấm lại sau epoch chỉ 0.444. Hai số khác nhau vì chúng đo hai trạng thái trọng số khác nhau: một bên là trung bình trong lúc model đang học, một bên là model ở cuối epoch.

Checkpoint được chọn theo validation loss, chứ không theo accuracy. Với bốn mẫu, accuracy chỉ nhảy theo từng nấc 0.25, nên từ epoch 5 trở đi nó luôn bằng 1. Loss thì vẫn tiếp tục giảm, và lần này epoch 30 là tốt nhất. Validation loss còn đang giảm ở epoch cuối là dấu hiệu có thể train thêm. `deepcopy` là cần thiết, vì nếu chỉ gán thẳng `state_dict()`, các epoch sau sẽ ghi đè lên checkpoint đã lưu.

Để kiểm việc khôi phục, ta lưu vào một vùng nhớ thay cho file, rồi nạp vào một model hoàn toàn mới. Gói lưu chứa những thứ cần cho suy luận: trọng số, mean và std để chuẩn hóa, thứ tự các feature, và tên các lớp. Không cần lưu optimizer, vì ở bước này ta không học tiếp.

In [ ]:
model.eval()
with torch.no_grad():
    expected = model(valid_x)
packet = {"state": best_state, "mean": mean, "scale": scale,
          "features": ["x1", "x2"], "classes": ["không", "có"]}
buffer = io.BytesIO()
torch.save(packet, buffer)
buffer.seek(0)
restored = torch.load(buffer, map_location="cpu", weights_only=True)
fresh = nn.Linear(2, 2)
fresh.load_state_dict(restored["state"])
fresh.eval()
with torch.no_grad():
    new_x = (raw_valid - restored["mean"]) / restored["scale"]
    actual = fresh(new_x)
print("logits khôi phục khớp:", torch.allclose(expected, actual))
print("nhãn bốn mẫu mới:", actual.argmax(dim=1).tolist())
print("thứ tự feature:", restored["features"])
assert torch.allclose(expected, actual) and not actual.requires_grad

logits khôi phục khớp: True
nhãn bốn mẫu mới: [0, 0, 1, 1]
thứ tự feature: ['x1', 'x2']


Model mới, dùng mean và std lấy từ gói lưu, cho đúng các logit như model cũ. Lưu ý là ta chuẩn hóa lại từ dữ liệu thô `raw_valid`, đúng như khi gặp dữ liệu mới thật. Ở đây việc khôi phục diễn ra trong cùng một chương trình. Khi triển khai thật, chương trình suy luận phải tự nạp gói, và đưa các cột vào đúng thứ tự đã lưu. Bốn mẫu này đã được dùng để chọn epoch, nên kết quả của chúng là validation, không phải test độc lập.

## Ví dụ 3: Batch cuối nhỏ không được tính ngang một batch đầy

Giả sử bốn mẫu có loss lần lượt là 1, 1, 1 và 9. DataLoader chia thành một batch 3 mẫu và một batch 1 mẫu. Thứ ta cần báo là loss trung bình trên bốn mẫu, không phải trung bình của hai batch.

In [ ]:
import torch
from torch.utils.data import DataLoader

torch.set_num_threads(1)
item_losses = torch.tensor([1., 1., 1., 9.])
loader = DataLoader(item_losses, batch_size=3, shuffle=False)
batch_means, total, seen = [], 0., 0
for losses in loader:
    mean_loss = losses.mean().item()
    batch_means.append(mean_loss)
    total += mean_loss * len(losses)
    seen += len(losses)
print("mean từng batch:", batch_means)
print("mean ngang batch / theo mẫu:", sum(batch_means) / len(batch_means), total / seen)
assert total / seen == item_losses.mean().item() == 3.

mean từng batch: [1.0, 9.0]
mean ngang batch / theo mẫu: 5.0 3.0


Trung bình đúng là $(1+1+1+9)/4=3$. Lấy trung bình của hai batch cho ra 5, vì một mẫu lẻ ở batch cuối được tính nặng ngang ba mẫu của batch đầu. Cách sửa là nhân loss của mỗi batch với số mẫu trong batch, cộng dồn, rồi chia cho tổng số mẫu. Cách này đúng khi loss là trung bình đều trên các mẫu. Nếu loss có trọng số theo lớp, có bỏ qua một số nhãn bằng `ignore_index`, hay tính theo từng pixel, thì mẫu số phải đếm theo đúng định nghĩa của loss đó.